# Задача 1. Compare

Реализовать рекурсивные версии mergesort и quicksort. Реализовать декоратор, который замеряет время выполнения функции. Придумать тесты, на которых время выполнения этих методов заметно различается.

## Идея

Mergesort делит массив пополам, рекурсивно сортирует обе части и сливает их в один отсортированный массив. Quicksort выбирает последний элемент опорным, ставит элементы не больше него слева, остальные справа и рекурсивно сортирует две стороны. Обе функции возвращают новый список и не меняют входной.

Декоратор запоминает время перед вызовом и после него. Результат функции остаётся прежним, а длительность последнего вызова доступна через `last_time`. На уже отсортированном массиве выбранный вариант quicksort каждый раз отделяет только один элемент. У mergesort деление остаётся примерно пополам — это подходящий случай для сравнения.

In [1]:
from functools import wraps
from time import perf_counter


def measure_time(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = function(*args, **kwargs)
        wrapper.last_time = perf_counter() - start
        return result

    wrapper.last_time = None
    return wrapper


@measure_time
def mergesort(nums):
    def sort(part):
        if len(part) <= 1:
            return part

        middle = len(part) // 2
        left = sort(part[:middle])
        right = sort(part[middle:])
        merged = []
        i = j = 0

        while i < len(left) and j < len(right):
            if left[i] <= right[j]:
                merged.append(left[i])
                i += 1
            else:
                merged.append(right[j])
                j += 1

        merged.extend(left[i:])
        merged.extend(right[j:])
        return merged

    return sort(nums[:])


@measure_time
def quicksort(nums):
    result = nums[:]

    def partition(left, right):
        pivot = result[right]
        border = left
        for i in range(left, right):
            if result[i] <= pivot:
                result[border], result[i] = result[i], result[border]
                border += 1
        result[border], result[right] = result[right], result[border]
        return border

    def sort(left, right):
        if left >= right:
            return
        pivot_index = partition(left, right)
        sort(left, pivot_index - 1)
        sort(pivot_index + 1, right)

    sort(0, len(result) - 1)
    return result

## Пошаговый пример

Для `[4, 1, 3, 2]` mergesort делит массив на `[4, 1]` и `[3, 2]`, затем на одиночные элементы. На обратном пути получает `[1, 4]` и `[2, 3]`, а при последнем слиянии — `[1, 2, 3, 4]`.

У quicksort первый опорный элемент в этом примере — последний, `2`. После разделения получаем `[1, 2, 3, 4]`: слева от `2` стоит `1`, справа — `3, 4`. Остаётся рекурсивно обработать правую часть. На отсортированном входе, напротив, последний элемент каждый раз максимален, поэтому правая часть пуста, а левая укорачивается лишь на один элемент.

## Сложность

Mergesort: `O(n log n)` времени в любом случае и `O(n)` дополнительной памяти для частей и слияния. Quicksort: в среднем `O(n log n)` времени и `O(log n)` памяти на рекурсию, но с последним элементом в роли опорного на уже отсортированном массиве — `O(n²)` времени и `O(n)` памяти на рекурсию. Обе реализации также создают копию входного массива `O(n)`; если учитывать эту копию, полная дополнительная память quicksort — `O(n)` даже в среднем.

Декоратор добавляет `O(1)` времени и памяти на один внешний вызов. Числа в замере зависят от компьютера, поэтому тест проверяет корректность, а не фиксированный порог времени.

## Тесты и сравнение времени

Проверяем пустой список, один элемент, повторы, отрицательные числа, уже отсортированный и обратный порядок. Затем три раза замеряем оба метода на отсортированном массиве и выводим лучший результат каждого метода. Размер `600` выбран так, чтобы рекурсивный quicksort не упёрся в стандартный лимит глубины Python.

In [2]:
cases = [
    [], [7], [4, 1, 3, 2], [5, 5, 5, 5],
    [-3, 8, 0, -1, 8], list(range(20)), list(range(20, 0, -1))
]
for case in cases:
    expected = sorted(case)
    before = case[:]
    assert mergesort(case) == expected
    assert quicksort(case) == expected
    assert case == before
    assert mergesort.last_time >= 0
    assert quicksort.last_time >= 0

assert mergesort.__name__ == "mergesort"
assert quicksort.__name__ == "quicksort"

ordered = list(range(600))
merge_times = []
quick_times = []
for _ in range(3):
    assert mergesort(ordered) == ordered
    merge_times.append(mergesort.last_time)
    assert quicksort(ordered) == ordered
    quick_times.append(quicksort.last_time)

print("Упорядоченный массив из 600 элементов:")
print(f"mergesort: {min(merge_times):.6f} с")
print(f"quicksort: {min(quick_times):.6f} с")
print("Все тесты пройдены")

Упорядоченный массив из 600 элементов:
mergesort: 0.000636 с
quicksort: 0.012377 с
Все тесты пройдены
